# Final Project: Modern Statistical Data Analysis 52311
Omer Sutovsky and Ido Ravid

In [1]:
# Imports
import os, glob, re, warnings
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.base import BaseEstimator, clone
from sklearn.decomposition import PCA
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr
from scipy import stats
import contextily as ctx
from pyproj import Transformer
from bidi.algorithm import get_display
from PIL import Image

_CODE_DIR       = os.getcwd()
DATA_DIR        = os.path.normpath(os.path.join(_CODE_DIR, '..', 'data'))
PLOTS_DIR_SUP   = os.path.normpath(os.path.join(_CODE_DIR, '..', 'plots', 'supervised'))
PLOTS_DIR_UNSUP = os.path.normpath(os.path.join(_CODE_DIR, '..', 'plots', 'unsupervised'))
PLOTS_DIR_HYP   = os.path.normpath(os.path.join(_CODE_DIR, '..', 'plots', 'hypothesis'))
os.makedirs(PLOTS_DIR_SUP, exist_ok=True)
os.makedirs(PLOTS_DIR_UNSUP, exist_ok=True)
os.makedirs(PLOTS_DIR_HYP, exist_ok=True)
warnings.simplefilter("ignore", pd.errors.PerformanceWarning)


---
## Section 1 - Supervised Learning

### Q1 - Preprocessing


In [2]:
########## Auxiliary - supervised learning ##########

SUPERVISED_DIR = os.path.normpath(os.path.join(DATA_DIR, 'supervised'))
os.makedirs(SUPERVISED_DIR, exist_ok=True)

RAW_SUPERVISED_FILE = os.path.join(DATA_DIR, 'H20121242Data.csv')
LABELED_PATH = os.path.join(SUPERVISED_DIR, 'supervised_labeled.csv')
FEATURE_DICTIONARY_PATH = os.path.join(SUPERVISED_DIR, 'feature_dictionary.csv')


def safe_to_csv(df, path, index=False):
    try:
        df.to_csv(path, index=index, encoding='utf-8')
    except PermissionError:
        print(f'Could not overwrite {path}; it may be open in another program. Continuing.')


def f1_score_binary(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    denom = 2 * tp + fp + fn
    return 0.0 if denom == 0 else float(2 * tp / denom)


def safe_name(value):
    value = str(value).strip()
    value = value.replace("+", "plus")
    value = re.sub(r"[^0-9A-Za-z_]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value or "missing"

def numeric_code_series(source):
    return pd.to_numeric(df[source], errors="coerce")

def register_feature(name, source_columns, transformation, feature_type, notes=""):
    if isinstance(source_columns, str):
        source_columns = [source_columns]
    feature_rows.append({
        "feature_name": name,
        "source_columns": ",".join(source_columns),
        "feature_type": feature_type,
        "transformation": transformation,
        "notes": notes,
    })
    handled_sources.update(source_columns)

def add_feature(name, values, source_columns, transformation, feature_type="numeric", notes="", is_score=False):
    features[name] = pd.to_numeric(values, errors="coerce")
    register_feature(name, source_columns, transformation, feature_type, notes)
    if is_score:
        score_feature_names.append(name)

def add_flag(name, values, source_columns, transformation, notes=""):
    features[name] = pd.Series(values, index=df.index).fillna(False).astype(int)
    register_feature(name, source_columns, transformation, "indicator", notes)

def add_one_hot(source, prefix=None, transformation="one-hot categorical encoding", labels=None, include_missing=True, notes=""):
    prefix = prefix or source
    s = numeric_code_series(source)
    cats = pd.Series(index=df.index, dtype="object")
    for idx, val in s.items():
        if pd.isna(val):
            cats.loc[idx] = "missing" if include_missing else np.nan
        else:
            key = int(val) if float(val).is_integer() else val
            label = labels.get(key, f"code_{safe_name(key)}") if labels else f"code_{safe_name(key)}"
            cats.loc[idx] = safe_name(label)
    dummies = pd.get_dummies(cats, prefix=prefix, dtype=int)
    for col in dummies.columns:
        features[col] = dummies[col]
        register_feature(col, source, transformation, "indicator", notes)

def add_mapped_score(source, name, mapping, transformation, idk_codes=None, not_relevant_codes=None, notes=""):
    s = numeric_code_series(source)
    score = s.map(mapping)
    add_feature(name, score, source, transformation, "semantic_score", notes, is_score=True)
    add_flag(f"{name}_missing", s.isna(), source, f"missing indicator for {source}")
    unmapped = s.notna() & score.isna()
    if unmapped.any():
        add_flag(f"{name}_unmapped", unmapped, source, f"non-missing value not found in score map for {source}")
    if idk_codes:
        flag_name = f"{name}_idk"
        add_flag(flag_name, s.isin(idk_codes), source, f"IDK indicator for {source}")
        idk_feature_names.append(flag_name)
    if not_relevant_codes:
        add_flag(f"{name}_not_relevant", s.isin(not_relevant_codes), source, f"not-relevant indicator for {source}")

def add_binary_yes_no(source, prefix=None, notes="1=yes, 2=no"):
    prefix = prefix or source
    s = numeric_code_series(source)
    add_feature(f"{prefix}_yes", s.map({1: 1, 2: 0}), source, "binary yes/no score", "semantic_score", notes, is_score=True)
    add_flag(f"{prefix}_missing", s.isna(), source, f"missing indicator for {source}")

def add_yes_no_idk_indicators(source, prefix=None, notes="1=yes, 2=no, 3=IDK"):
    prefix = prefix or source
    s = numeric_code_series(source)
    add_flag(f"{prefix}_yes", s.eq(1), source, "yes indicator", notes)
    add_flag(f"{prefix}_no", s.eq(2), source, "no indicator", notes)
    idk_name = f"{prefix}_idk"
    add_flag(idk_name, s.eq(3), source, "IDK indicator", notes)
    add_flag(f"{prefix}_missing", s.isna(), source, "missing indicator", notes)
    idk_feature_names.append(idk_name)

def add_open_bin_score(source, name, mapping, top_codes, unknown_codes=None, transformation="mapped binned numeric value"):
    s = numeric_code_series(source)
    unknown_codes = set(unknown_codes or [])
    clean = s.mask(s.isin(unknown_codes))
    add_feature(name, clean.map(mapping), source, transformation, "numeric", "open-ended bins use extrapolated midpoint", is_score=True)
    add_flag(f"{name}_missing", s.isna(), source, f"missing indicator for {source}")
    add_flag(f"{name}_top_bin", s.isin(top_codes), source, f"top/open bin indicator for {source}")
    for code in unknown_codes:
        add_flag(f"{name}_code_{int(code)}", s.eq(code), source, f"special code {int(code)} indicator for {source}")

def add_mean_feature(name, cols, notes):
    existing = [c for c in cols if c in features.columns]
    if existing:
        add_feature(name, features[existing].mean(axis=1, skipna=True), existing, "row-wise mean of related semantic scores", "aggregate_score", notes, is_score=True)


In [3]:
## Step 1 - preprocessing
df = pd.read_csv(RAW_SUPERVISED_FILE, encoding="cp1255")

features = pd.DataFrame(index=df.index)
feature_rows = []
handled_sources = set()
idk_feature_names = []
score_feature_names = []

# these columns leak information or include text, so we remove them
LEAKAGE_COLUMNS = {"Hovot", "Hovot2", "Hovot3", "Hovot4", "Hovot5", "Hovot6", "Hovot7", "Hovot8", "Hovot9", "HovotLechatzim"}
TEXT_COLUMNS = ["OverVaI7", "Halvaa10", "Hovot9", "Hashkaot10", "HipusM9", "MekorotM10"]
TEXT_VALID_CODE_RANGES = {
    "OverVaI7": {1, 2},
    "Halvaa10": {1, 2, 3},
    "Hovot9": {1, 2, 3, 4, 5},
    "Hashkaot10": {1, 2, 3},
    "HipusM9": {1, 2, 3, 4, 5},
    "MekorotM10": {1, 2, 3},
}
NON_FEATURE_COLUMNS = {"SerialNumber", "Shana", "nn"}

# adding the label: Hovot=1 is no debt (0), Hovot=2 has debt (1)
y = df["Hovot"].map({1: 0, 2: 1})
labeled_mask = y.notna()
unlabeled_mask = y.isna()

In [4]:
# demographic engineered variables
add_one_hot("Machoz", "district")
add_open_bin_score("Ms_Nefashot", "household_size_mid", {1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7.5}, top_codes=[7])
add_one_hot("Minn", "gender")
add_open_bin_score("Gil", "age_mid", {1: 23, 2: 28, 3: 33, 4: 38, 5: 43, 6: 48, 7: 53, 8: 58, 9: 63, 10: 70, 11: 80}, top_codes=[11])
add_one_hot("MatzavMishp", "family_status")
add_one_hot("Pop_Group", "population_group")
add_one_hot("DatSafa_C", "religion_language")
add_one_hot("HesderDiyur", "living_arrangement")
add_open_bin_score("ShnotLimudKlali_C", "education_years_mid", {1: 2.5, 2: 6.5, 3: 9.5, 4: 11.5, 5: 14, 6: 17}, top_codes=[6])
add_one_hot("MatzavTaasuka_C", "employment_status")
add_open_bin_score("HachnasaKolelet", "household_income_mid", {1: 1250, 2: 3250.5, 3: 4500.5, 4: 5750.5, 5: 7250.5, 6: 9000.5, 7: 11500.5, 8: 15000.5, 9: 20500.5, 10: 27500.5, 11: 0}, top_codes=[10], unknown_codes=[98, 99])
add_open_bin_score("HachnasaAvoda", "work_income_mid", {1: 1000, 2: 2500.5, 3: 3500.5, 4: 4500.5, 5: 5500.5, 6: 6750.5, 7: 8750.5, 8: 12000.5, 9: 17500.5, 10: 24500.5, 11: 0}, top_codes=[10], unknown_codes=[98, 99])
add_one_hot("SemelMishlach_wp", "profession")

In [5]:
## features we extract from the quiz

# Q1,Q3,Q11,Q26: yes/no answers.
for col in ["OverVashav", "OverVaI1", "OverVaI2", "OverVaI3", "OverVaI4", "OverVaI5", "OverVaI6", "Mashkanta1", "NoseCaspiHarhavaNihul", "NoseCaspiHarhavaCarhanut", "NoseCaspiHarhavaHalvaot", "NoseCaspiHarhavaHishonot", "NoseCaspiHarhavaBituhim", "NoseCaspiHarhavaMashkanta", "NoseCaspiHarhavaEynCoreh"]:
    add_binary_yes_no(col)

# Q2 bank balance tracking: 2 > 3 > 4 > 5 > 1.
add_mapped_score("OverVaMaakav", "bank_balance_tracking_score", {2: 4, 3: 3, 4: 2, 5: 1, 1: 0}, "ordinal score: weekly+ tracking highest, no tracking lowest")

# Q4,Q8,Q10,Q13,Q16,Q19: yes/no/IDK
for col in ["OverVaAcher", "KartisAshP1", "KartisAshP2", "KartisAshP3", "KartisAshP4", "KartisAshS1", "KartisAshS2", "KartisAshS3", "KartisAshS4", "KartisAshS5", "KartisAshS6", "KartisAshS7", "KartisAshNechsam", "Halvaa2", "Halvaa3", "Halvaa4", "Halvaa5", "Halvaa6", "Halvaa7", "Halvaa8", "Halvaa9", "Hashkaot1", "Hashkaot2", "Hashkaot3", "Hashkaot4", "Hashkaot5", "Hashkaot6", "Hashkaot7", "Hashkaot8", "Hashkaot9", "SherutY1", "SherutY2", "SherutY3", "SherutY4", "SherutY5", "SherutY6"]:
    add_yes_no_idk_indicators(col)

# Q5 banking events: 1 good, 5 bad.
for col in ["OverVaEruim1", "OverVaEruim2", "OverVaEruim3"]:
    add_mapped_score(col, f"{col}_frequency_good_behavior_score", {1: 4, 2: 3, 3: 2, 4: 1, 5: 0}, "frequency score: never is best, usual state is worst")

# Q6,Q17: categorical, with IDK as a signal
for col in ["HeshbonMaakav", "HashkaotMaclit"]:
    add_one_hot(col, col, "one-hot decision-maker category")
    s = numeric_code_series(col)
    flag_name = f"{col}_idk"
    add_flag(flag_name, s.eq(8), col, "decision-maker IDK indicator")
    idk_feature_names.append(flag_name)

# Q7 credit cards: we map 1->0, ..., 5->4+ with extrapolated top value.
add_open_bin_score("KartisAshKama", "credit_cards_count", {1: 0, 2: 1, 3: 2, 4: 3, 5: 4.5}, top_codes=[5], transformation="credit-card count from coded answers")

# Q12: categorical, with IDK and no-morgatge as signals
add_one_hot("MashkantaMekorot", "mortgage_info_source", "one-hot mortgage information source")
add_flag("mortgage_info_source_idk", numeric_code_series("MashkantaMekorot").eq(7), "MashkantaMekorot", "mortgage-source IDK indicator")
idk_feature_names.append("mortgage_info_source_idk")
add_flag("mortgage_info_source_not_app_no_mortgage", numeric_code_series("Mashkanta1").eq(2) & numeric_code_series("MashkantaMekorot").isna(), ["Mashkanta1", "MashkantaMekorot"], "structural missingness: no mortgage, so source not applicable")

# Q13 loan yes/no.
add_binary_yes_no("Halvaa", "has_loan")

# Q18 insurance: IDK is signal
for col in ["Bituhim1", "Bituhim2", "Bituhim3", "Bituhim4", "Bituhim5", "Bituhim6"]:
    add_one_hot(col, col, "one-hot insurance answer: yes/no/IDK/no need")
    s = numeric_code_series(col)
    idk_name = f"{col}_idk_strong_signal"
    add_flag(idk_name, s.eq(3), col, "IDK is a strong insurance uncertainty signal")
    idk_feature_names.append(idk_name)

# Q20 financial knowledge:  since yes is correct,
# we treat it as: yes > IDK > no.
for col in ["NosCas1", "NosCas2", "NosCas3", "NosCas4"]:
    add_mapped_score(col, f"{col}_knowledge_score", {1: 2, 3: 1, 2: 0}, "financial knowledge score: yes correct, IDK middle, no lowest", idk_codes=[3])

# Q21 interest calculation numeric/free answer, we use binnning.
s21 = numeric_code_series("HishuvRibit")
q21_score = pd.Series(np.nan, index=df.index, dtype="float64")
q21_score[s21.eq(1020)] = 4
q21_score[(s21 >= 1000) & (s21 <= 1500) & ~s21.eq(1020)] = 3
q21_score[s21.eq(2)] = 2
q21_score[s21 > 1500] = 1
q21_score[(s21 < 1000) & ~s21.eq(2)] = 0
add_feature("interest_calc_1_score", q21_score, "HishuvRibit", "Q21 score: 1020 highest; 1000-1500 good; IDK middle; >1500 low; <1000 lowest", "semantic_score", is_score=True)
add_flag("interest_calc_1_idk", s21.eq(2), "HishuvRibit", "Q21 IDK code indicator")
add_flag("interest_calc_1_missing", s21.isna(), "HishuvRibit", "Q21 missing indicator")
idk_feature_names.append("interest_calc_1_idk")

# Q22,Q23 financial knowledge scores.
add_mapped_score("HishuvRibitDribit", "interest_calc_2_score", {1: 5, 2: 4, 3: 3, 5: 2, 4: 1}, "Q22 score order: 1 > 2 > 3 > 5 > 4", idk_codes=[5])
add_mapped_score("ShiurRibitBank", "saving_program_interest_score", {2: 4, 3: 3, 1: 2, 4: 1}, "Q23 score order: 2 > 3 > 1 > 4", idk_codes=[4])

# Q24 financial information search, whrere we tret higher as better.
for col in ["HipusM1", "HipusM2", "HipusM3", "HipusM4", "HipusM5", "HipusM6", "HipusM7", "HipusM8"]:
    add_mapped_score(col, f"{col}_info_search_score", {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}, "Q24 information-search frequency score: higher is better")

# Q25: removed code 5 since its not relevant.
positive_q25 = [1, 3, 5, 6, 7, 8, 10]
negative_q25 = [2, 4, 9]
for i in positive_q25:
    add_mapped_score(f"NoseCaspiM{i}", f"NoseCaspiM{i}_attitude_score", {1: 4, 2: 3, 3: 2, 4: 1, 5: 0}, "Q25 positive statement score: strongly agree highest; not relevant zero", not_relevant_codes=[5])
for i in negative_q25:
    add_mapped_score(f"NoseCaspiM{i}", f"NoseCaspiM{i}_attitude_score", {1: 1, 2: 2, 3: 3, 4: 4, 5: 0}, "Q25 negative statement score: strongly disagree highest; not relevant zero", not_relevant_codes=[5])

# Q26 aggregate: 1-6 is the same, 7 different signal
interest_cols = ["NoseCaspiHarhavaNihul", "NoseCaspiHarhavaCarhanut", "NoseCaspiHarhavaHalvaot", "NoseCaspiHarhavaHishonot", "NoseCaspiHarhavaBituhim", "NoseCaspiHarhavaMashkanta"]
interest_yes = pd.DataFrame({col: numeric_code_series(col).eq(1).astype(int) for col in interest_cols})
add_feature("knowledge_expansion_topics_count", interest_yes.sum(axis=1), interest_cols, "count of selected knowledge-expansion topics among fields 1-6", "aggregate_count")
add_flag("knowledge_expansion_no_need", numeric_code_series("NoseCaspiHarhavaEynCoreh").eq(1), "NoseCaspiHarhavaEynCoreh", "selected no need to expand financial knowledge")

# Q27 information source - indicators
for col in ["MekorotM1", "MekorotM2", "MekorotM3", "MekorotM4", "MekorotM5", "MekorotM6", "MekorotM7", "MekorotM8", "MekorotM9"]:
    add_one_hot(col, col, "Q27 non-ordinal source-interest one-hot")
    s = numeric_code_series(col)
    idk_name = f"{col}_idk"
    add_flag(idk_name, s.eq(3), col, "Q27 source-interest IDK indicator")
    idk_feature_names.append(idk_name)

# Q28 preferred source - one hot
add_one_hot("MakorMuadaf", "preferred_info_source", "Q28 preferred information source one-hot")
add_one_hot("MakorMahooM", "preferred_source_number", "Q28 selected source number one-hot")

# Q29 mortgage knowledge: 1 best -> 5 worst
add_mapped_score("RamatYedaMash", "mortgage_knowledge_score", {1: 4, 2: 3, 3: 2, 4: 1, 5: 0}, "Q29 mortgage knowledge score: 1 highest, 5 lowest")

In [6]:
# IDK count and rate across all survey answers
if idk_feature_names:
    idk_matrix = features[idk_feature_names]
    add_feature("idk_total", idk_matrix.sum(axis=1), idk_feature_names, "total number of IDK indicators across engineered features", "aggregate_count")
    add_feature("idk_rate", idk_matrix.mean(axis=1), idk_feature_names, "share of IDK indicators across engineered features", "aggregate_rate")

add_mean_feature("q20_knowledge_mean", [f"NosCas{i}_knowledge_score" for i in range(1, 5)], "Q20 financial knowledge mean")
add_mean_feature("q24_info_search_mean", [f"HipusM{i}_info_search_score" for i in range(1, 9)], "Q24 information search mean")
add_mean_feature("q25_attitude_mean", [f"NoseCaspiM{i}_attitude_score" for i in range(1, 11)], "Q25 financial attitude mean")
add_mean_feature("financial_knowledge_mean", ["q20_knowledge_mean", "interest_calc_1_score", "interest_calc_2_score", "saving_program_interest_score", "mortgage_knowledge_score"], "combined financial knowledge proxy")

# catch any remaining non-excluded raw columns
excluded_sources = LEAKAGE_COLUMNS | set(TEXT_COLUMNS) | NON_FEATURE_COLUMNS
remaining = [c for c in df.columns if c not in handled_sources and c not in excluded_sources]
for col in remaining:
    add_one_hot(col, col, "fallback one-hot for remaining non-excluded source column", notes="Added so no usable raw column is silently ignored.")


# fill remaining NaNs with labeled-row medians
filled_features = features.copy()
fill_values = filled_features.loc[labeled_mask].median(numeric_only=True).fillna(0)
filled_features = filled_features.fillna(fill_values).fillna(0)
assert not filled_features.isna().any().any(), "Feature matrix still contains NaN after neutral fill"

feature_dict = pd.DataFrame(feature_rows)


In [7]:
########## Auxiliary - supervised learning ##########

# build outputs
labeled_out = pd.concat([
    df.loc[labeled_mask, ["SerialNumber"]].reset_index(drop=True),
    y.loc[labeled_mask].astype(int).rename("y_debt").reset_index(drop=True),
    df.loc[labeled_mask, "nn"].rename("sample_weight").reset_index(drop=True),
    filled_features.loc[labeled_mask].reset_index(drop=True),
], axis=1)

unlabeled_out = pd.concat([
    df.loc[unlabeled_mask, ["SerialNumber"]].reset_index(drop=True),
    df.loc[unlabeled_mask, "Hovot"].rename("raw_Hovot").reset_index(drop=True),
    df.loc[unlabeled_mask, "nn"].rename("sample_weight").reset_index(drop=True),
    filled_features.loc[unlabeled_mask].reset_index(drop=True),
], axis=1)

labeled_out.to_csv(os.path.join(SUPERVISED_DIR, 'supervised_labeled.csv'), index=False, encoding="utf-8")
unlabeled_out.to_csv(os.path.join(SUPERVISED_DIR, 'supervised_unlabeled_hovot_missing.csv'), index=False, encoding="utf-8")
feature_dict.to_csv(os.path.join(SUPERVISED_DIR, 'feature_dictionary.csv'), index=False, encoding="utf-8")

print("Labeled shape:", labeled_out.shape)
print("Debt label counts:")
print(labeled_out["y_debt"].value_counts().sort_index())

text_df = df[TEXT_COLUMNS].copy()
text_cleanup_records = []
for col, valid_codes in TEXT_VALID_CODE_RANGES.items():
    stripped = text_df[col].dropna().astype(str).str.strip()
    numeric_code = stripped.str.fullmatch(r"\d+")
    numeric_values = pd.to_numeric(stripped.where(numeric_code), errors="coerce")
    valid_numeric_code = numeric_code & numeric_values.isin(valid_codes)
    for idx, value in stripped[valid_numeric_code].items():
        text_cleanup_records.append({
            "SerialNumber": df.loc[idx, "SerialNumber"],
            "source_column": col,
            "removed_value": value,
            "reason": "single numeric answer code in free-text detail field",
        })
    text_df.loc[stripped[valid_numeric_code].index, col] = np.nan

text_records = []
for col in TEXT_COLUMNS:
    non_missing = text_df[col].notna()
    for idx, value in text_df.loc[non_missing, col].items():
        text_records.append({
            "SerialNumber": df.loc[idx, "SerialNumber"],
            "source_column": col,
            "raw_value": value,
        })
free_text_audit = pd.DataFrame(text_records, columns=["SerialNumber", "source_column", "raw_value"])
free_text_numeric_code_cleanup = pd.DataFrame(text_cleanup_records, columns=["SerialNumber", "source_column", "removed_value", "reason"])

free_text_audit.to_csv(os.path.join(SUPERVISED_DIR, 'free_text_audit.csv'), index=False, encoding="utf-8")
free_text_numeric_code_cleanup.to_csv(os.path.join(SUPERVISED_DIR, 'free_text_numeric_code_cleanup.csv'), index=False, encoding="utf-8")

Labeled shape: (1153, 447)
Debt label counts:
y_debt
0    923
1    230
Name: count, dtype: int64


### Q2 - Lasso Path


In [8]:
########## Auxiliary - supervised learning ##########
COEFFICIENTS_PATH = os.path.join(SUPERVISED_DIR, 'lasso_path_coefficients.csv')
TOP_FEATURES_PATH = os.path.join(SUPERVISED_DIR, 'lasso_path_top_features.csv')
LASSO_PLOT_PATH = os.path.join(PLOTS_DIR_SUP, 'lasso_path.png')


In [9]:
## Step 2 - L1 logistic path

def run_lasso_path():
    data = pd.read_csv(LABELED_PATH)
    feature_dictionary = pd.read_csv(FEATURE_DICTIONARY_PATH)
    feature_names = [c for c in data.columns if c not in {"SerialNumber", "y_debt", "sample_weight"}]
    X_raw = data[feature_names].to_numpy(dtype=float)
    y = data["y_debt"].to_numpy(dtype=int)

    if os.path.exists(COEFFICIENTS_PATH) and os.path.exists(TOP_FEATURES_PATH):
        return {"coefficients": pd.read_csv(COEFFICIENTS_PATH), "top_features": pd.read_csv(TOP_FEATURES_PATH)}

    scaler = StandardScaler()
    X_scaled_all = scaler.fit_transform(X_raw)
    non_constant = scaler.scale_ > 1e-12  # drop zero-variance features before fitting
    X = X_scaled_all[:, non_constant]
    kept_features = np.array(feature_names, dtype=object)[non_constant]

    # log grid from strong -> weak regularization
    C_values = np.geomspace(0.05, 30.0, 35)
    coefs = []
    intercepts = []
    active_counts = []
    model = LogisticRegression(
        penalty="l1",
        solver="liblinear",
        C=float(C_values[0]),
        max_iter=5000,
        class_weight=None,
        warm_start=True,
        random_state=42,
    )
    for C in C_values:
        model.C = float(C)
        model.fit(X, y)
        beta = model.coef_.ravel()
        coefs.append(beta)
        intercepts.append(float(model.intercept_[0]))
        active_counts.append(int(np.abs(beta).gt(1e-10).sum()) if hasattr(np.abs(beta), "gt") else int((np.abs(beta) > 1e-10).sum()))

    coef_path = np.vstack(coefs)
    active = np.abs(coef_path) > 1e-10
    entry_steps = np.where(active.any(axis=0), active.argmax(axis=0) + 1, np.inf)
    max_abs = np.abs(coef_path).max(axis=0)

    source_lookup = feature_dictionary.drop_duplicates("feature_name").set_index("feature_name")
    top_rows = []
    for j, feature in enumerate(kept_features):
        top_rows.append({
            "feature_name": feature,
            "max_abs_standardized_coef": float(max_abs[j]),
            "final_standardized_coef": float(coef_path[-1, j]),
            "entry_C": np.nan if not np.isfinite(entry_steps[j]) else float(C_values[int(entry_steps[j]) - 1]),
            "entry_step": np.nan if not np.isfinite(entry_steps[j]) else int(entry_steps[j]),
            "source_columns": source_lookup["source_columns"].get(feature, ""),
            "feature_type": source_lookup["feature_type"].get(feature, ""),
            "transformation": source_lookup["transformation"].get(feature, ""),
        })
    top_features = pd.DataFrame(top_rows).query("max_abs_standardized_coef > 0").sort_values(
        ["max_abs_standardized_coef", "entry_step"], ascending=[False, True]
    ).reset_index(drop=True)

    path_df = pd.DataFrame(coef_path, columns=kept_features)
    path_df.insert(0, "active_count", active_counts)
    path_df.insert(0, "intercept", intercepts)
    path_df.insert(0, "log_C", np.log(C_values))
    path_df.insert(0, "C", C_values)

    safe_to_csv(path_df, COEFFICIENTS_PATH)
    safe_to_csv(top_features, TOP_FEATURES_PATH)

    return {"coefficients": path_df, "top_features": top_features}

In [ ]:
# this run takes time
lasso_results = run_lasso_path()
lasso_top_features = lasso_results['top_features']
lasso_top_features.head(20)


/Users/idoravid/.local/share/mise/installs/python/3.12.11/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/idoravid/.local/share/mise/installs/python/3.12.11/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/idoravid/.local/share/mise/installs/python/3.12.11/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' i

In [ ]:
########## Auxiliary - supervised learning ##########

def plot_lasso_path(results):
    path_df = results["coefficients"]
    top_features = results["top_features"]
    C_values = path_df["C"].values
    kept_features = [c for c in path_df.columns if c not in {"C", "log_C", "intercept", "active_count"}]
    coef_path = path_df[kept_features].values
    top_plot = top_features.head(15)["feature_name"].tolist()
    plot_idx = [list(kept_features).index(name) for name in top_plot]
    plt.figure(figsize=(13, 7))
    for idx, name in zip(plot_idx, top_plot):
        plt.plot(np.log(C_values), coef_path[:, idx], linewidth=2, label=name)
    plt.axhline(0, color="black", linewidth=0.8)
    plt.xlabel("log(C), weaker regularization to the right")
    plt.ylabel("standardized coefficient")
    plt.title("L1 logistic-regression path for debt classification")
    plt.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8)
    plt.tight_layout()
    plt.savefig(LASSO_PLOT_PATH, dpi=140)
    plt.close()
    print("Features used:", len(kept_features))
    print("Active coefficients at final C:", int(path_df["active_count"].iloc[-1]))
    print(top_features.head(15)[["feature_name", "entry_step", "final_standardized_coef"]].to_string(index=False))

plot_lasso_path(lasso_results)

### Q3 - PCA


In [ ]:
########## Auxiliary - supervised learning ##########
PCA_EXPLAINED_PATH = os.path.join(SUPERVISED_DIR, 'pca_explained_variance.csv')
PCA_LOADINGS_PATH = os.path.join(SUPERVISED_DIR, 'pca_top_loadings.csv')
PCA_PR_AUC_PATH = os.path.join(SUPERVISED_DIR, 'pca_pr_auc.csv')
PCA_SCORES_PATH = os.path.join(SUPERVISED_DIR, 'pca_component_scores.csv')
PCA_LEAKAGE_AUDIT_PATH = os.path.join(SUPERVISED_DIR, 'pca_loading_leakage_audit.csv')

def infer_pca_source_context(source_columns):
    source = str(source_columns).split(",")[0].strip()
    prefix_contexts = [
        ("OverVa", 1, "before target Q14", "no target-branch leakage"),
        ("Heshbon", 6, "checking-account block before target Q14", "no target-branch leakage"),
        ("KartisAsh", 8, "credit-card block before target Q14", "no target-branch leakage"),
        ("Halvaa", 13, "loan block immediately before target Q14", "related debt proxy but not target-derived"),
        ("Hovot", 14, "target/debt-detail branch", "excluded leakage source; should not appear"),
        ("Hashkaot", 16, "investment block after target Q14", "post-target questionnaire block; not Hovot-derived"),
        ("Bituhim", 18, "insurance block after target Q14", "post-target questionnaire block; not Hovot-derived"),
        ("NosCas", 20, "knowledge block after target Q14", "post-target questionnaire block; not Hovot-derived"),
        ("NoseCaspi", 25, "attitude block after target Q14", "post-target questionnaire block; not Hovot-derived"),
        ("Mekorot", 27, "information-source interest block after target Q14", "post-target questionnaire block; not Hovot-derived"),
    ]
    for prefix, question, context, note in prefix_contexts:
        if source.startswith(prefix):
            return question, context, prefix == "Hovot", note
    return np.nan, "profile/background or aggregate source", False, "not target-branch leakage"

In [ ]:
## Step 3 - PCA

def run_pca_analysis_sklearn(n_components=4, top_loading_count=30):
    data = pd.read_csv(LABELED_PATH)
    feature_dictionary = pd.read_csv(FEATURE_DICTIONARY_PATH)
    feature_names = [c for c in data.columns if c not in {"SerialNumber", "y_debt", "sample_weight"}]
    X_raw = data[feature_names].to_numpy(dtype=float)
    y = data["y_debt"].to_numpy(dtype=int)

    scaler = StandardScaler()
    X_scaled_all = scaler.fit_transform(X_raw)
    non_constant = scaler.scale_ > 1e-12  # drop zero var features before PCA
    X = X_scaled_all[:, non_constant]
    kept_features = np.array(feature_names, dtype=object)[non_constant]

    pca = PCA(n_components=n_components)
    scores = pca.fit_transform(X)
    source_lookup = feature_dictionary.drop_duplicates("feature_name").set_index("feature_name")

    explained = pd.DataFrame({
        "component": [f"PC{i + 1}" for i in range(n_components)],
        "eigenvalue": pca.explained_variance_,
        "explained_variance_ratio": pca.explained_variance_ratio_,
        "cumulative_explained_variance_ratio": np.cumsum(pca.explained_variance_ratio_),
    })

    baseline = float(y.mean())  # how much positive-class is prevelent
    #  this is the PRAUC baseline for a random classifier
    pr_rows, loading_rows, audit_rows = [], [], []
    score_output = pd.DataFrame({"SerialNumber": data["SerialNumber"], "y_debt": y})

    for i in range(n_components):
        pc = f"PC{i + 1}"
        alpha = scores[:, i]
        pr_alpha = average_precision_score(y, alpha)
        pr_neg = average_precision_score(y, -alpha)
        # we check both + and - since its arbitrary in PCA
        best_orientation = "alpha" if pr_alpha >= pr_neg else "-alpha"
        best_pr = max(pr_alpha, pr_neg)
        score_output[f"{pc}_score_raw"] = alpha
        score_output[f"{pc}_score_oriented"] = alpha if best_orientation == "alpha" else -alpha
        pr_rows.append({
            "component": pc,
            "pr_auc_alpha": pr_alpha,
            "pr_auc_negative_alpha": pr_neg,
            "best_orientation": best_orientation,
            "best_pr_auc": best_pr,
            "baseline_positive_rate": baseline,
            "best_pr_auc_minus_baseline": best_pr - baseline,
        })

        loadings = pca.components_[i]
        order = np.argsort(-np.abs(loadings))  # rank features by size
        for rank, j in enumerate(order[:top_loading_count], start=1):
            feature = kept_features[j]
            row = {
                "component": pc,
                "rank": rank,
                "feature_name": feature,
                "abs_loading": abs(float(loadings[j])),
                "signed_loading": float(loadings[j]),
                "source_columns": source_lookup["source_columns"].get(feature, ""),
                "feature_type": source_lookup["feature_type"].get(feature, ""),
                "transformation": source_lookup["transformation"].get(feature, ""),
            }
            loading_rows.append(row)
            question, context, target_branch, note = infer_pca_source_context(row["source_columns"])
            audit_rows.append({**row, "approx_question": question, "question_context": context, "target_branch_source": target_branch, "leakage_audit_note": note})


    loadings_df = pd.DataFrame(loading_rows)
    pr_auc_df = pd.DataFrame(pr_rows)
    leakage_audit = pd.DataFrame(audit_rows)
    safe_to_csv(explained, PCA_EXPLAINED_PATH)
    safe_to_csv(loadings_df, PCA_LOADINGS_PATH)
    safe_to_csv(pr_auc_df, PCA_PR_AUC_PATH)
    safe_to_csv(score_output, PCA_SCORES_PATH)
    safe_to_csv(leakage_audit, PCA_LEAKAGE_AUDIT_PATH)

    return {"explained_variance": explained, "pr_auc": pr_auc_df, "top_loadings": loadings_df, "scores": score_output, "leakage_audit": leakage_audit}

In [ ]:
pca_results = run_pca_analysis_sklearn()
pca_results['pr_auc']


In [ ]:
########## Auxiliary - supervised learning ##########

def plot_pca_results(results):
    explained = results["explained_variance"]
    pr_auc_df = results["pr_auc"]
    loadings_df = results["top_loadings"]
    leakage_audit = results["leakage_audit"]
    print(explained.to_string(index=False))
    print(pr_auc_df.to_string(index=False))
    print("Target-branch rows among top loadings:", int(leakage_audit["target_branch_source"].sum()))
    for pc in explained["component"]:
        pc_loadings = loadings_df[loadings_df["component"] == pc].sort_values("rank")
        abs_vals = pc_loadings["abs_loading"].values
        plt.figure(figsize=(12, 6))
        plt.plot(abs_vals, linewidth=1.5)
        plt.title(f"{pc}: absolute PCA loadings, descending")
        plt.xlabel("features sorted by |loading|")
        plt.ylabel("absolute loading")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR_SUP, f'pca_{pc.lower()}_loadings.png'), dpi=140)
        plt.close()

plot_pca_results(pca_results)

### Q4 - Prediction Models


In [ ]:
########## Auxiliary - supervised learning ##########
SPLIT_PATH = os.path.join(SUPERVISED_DIR, 'model_split_assignments.csv')
CV_RESULTS_PATH = os.path.join(SUPERVISED_DIR, 'model_cv_results_part4.csv')
PERFORMANCE_PATH = os.path.join(SUPERVISED_DIR, 'model_performance_part4.csv')
PREDICTIONS_PATH = os.path.join(SUPERVISED_DIR, 'model_predictions_part4.csv')
LASSO_TOP15_PART4_PATH = os.path.join(SUPERVISED_DIR, 'model_lasso_top15_features_part4.csv')
F1_PLOT_PATH = os.path.join(PLOTS_DIR_SUP, 'part4_f1_scores.png')

SEED = 42
THRESHOLDS = np.round(np.arange(0.20, 0.61, 0.05), 2)


class ThresholdClassifier(BaseEstimator):
    def __init__(self, estimator=None, threshold=0.5):
        self.estimator = estimator
        self.threshold = threshold

    def fit(self, X, y):
        self.estimator_ = clone(self.estimator)
        self.estimator_.fit(X, y)
        return self

    def predict(self, X):
        if hasattr(self.estimator_, "predict_proba"):
            scores = self.estimator_.predict_proba(X)[:, 1]
        else:
            raw = self.estimator_.decision_function(X)
            scores = 1 / (1 + np.exp(-raw))
        return (scores >= self.threshold).astype(int)

    def predict_proba(self, X):
        if hasattr(self.estimator_, "predict_proba"):
            return self.estimator_.predict_proba(X)
        raw = self.estimator_.decision_function(X)
        scores = 1 / (1 + np.exp(-raw))
        return np.column_stack([1 - scores, scores])


def fit_grid(name, estimator, param_grid, X_train, y_train):
    grid = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring="f1",
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
        n_jobs=-1,
        refit=True,
        return_train_score=True,
    )
    grid.fit(X_train, y_train)
    rows = pd.DataFrame(grid.cv_results_)
    rows["model"] = name
    return grid, rows

In [ ]:
## Step 4 - prediction models

def build_split_and_features():
    data = pd.read_csv(LABELED_PATH)
    feature_names = [c for c in data.columns if c not in {"SerialNumber", "y_debt", "sample_weight"}]
    X = data[feature_names]
    y = data["y_debt"].astype(int)

    train_idx, test_idx = train_test_split(
        np.arange(len(data)),
        test_size=1/3,  # 2:1 train/test ratio
        random_state=SEED,
        stratify=y,  # preserve positive-class proportion in both splits
    )
    split = pd.DataFrame({"SerialNumber": data["SerialNumber"], "y_debt": y, "split": "test"})
    split.loc[train_idx, "split"] = "train"

    safe_to_csv(split, SPLIT_PATH)
    return data, X, y, np.sort(train_idx), np.sort(test_idx), feature_names


def select_top15_from_training_lasso(feature_names, X_train, y_train):
    feature_dictionary = pd.read_csv(FEATURE_DICTIONARY_PATH)
    source_lookup = feature_dictionary.drop_duplicates("feature_name").set_index("feature_name")

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train.to_numpy(dtype=float))
    non_constant = scaler.var_ > 1e-12
    kept_features = np.array(feature_names, dtype=object)[non_constant]
    X_path = X_scaled[:, non_constant]

    C_values = np.geomspace(0.05, 30.0, 35)
    coefs = []
    for C in C_values:
        model = LogisticRegression(
            penalty="l1",
            solver="liblinear",
            C=float(C),
            max_iter=5000,
            random_state=SEED,
        )
        model.fit(X_path, y_train)
        coefs.append(model.coef_.ravel())

    coef_path = np.vstack(coefs)
    active = np.abs(coef_path) > 1e-10
    entry_steps = np.where(active.any(axis=0), active.argmax(axis=0) + 1, np.inf)
    max_abs = np.abs(coef_path).max(axis=0)

    rows = []
    for j, feature in enumerate(kept_features):
        if max_abs[j] <= 0 or not np.isfinite(entry_steps[j]):
            continue
        rows.append({
            "feature_name": feature,
            "entry_step": int(entry_steps[j]),
            "entry_C": float(C_values[int(entry_steps[j]) - 1]),
            "max_abs_standardized_coef": float(max_abs[j]),
            "final_standardized_coef": float(coef_path[-1, j]),
            "source_columns": source_lookup["source_columns"].get(feature, ""),
            "feature_type": source_lookup["feature_type"].get(feature, ""),
            "transformation": source_lookup["transformation"].get(feature, ""),
        })

    ranked = pd.DataFrame(rows).sort_values(
        ["entry_step", "max_abs_standardized_coef"], ascending=[True, False]  # earlier entry = stronger L1 preference
    ).reset_index(drop=True)
    top15 = ranked.head(15).copy()
    top15.insert(0, "rank", np.arange(1, len(top15) + 1))
    safe_to_csv(top15, LASSO_TOP15_PART4_PATH)
    return top15["feature_name"].tolist()


def run_part4_models():
    # Skip recomputing if outputs already exist from a prior run
    if os.path.exists(CV_RESULTS_PATH) and os.path.exists(PERFORMANCE_PATH) and os.path.exists(PREDICTIONS_PATH):
        return {
            "performance": pd.read_csv(PERFORMANCE_PATH),
            "predictions": pd.read_csv(PREDICTIONS_PATH),
            "cv_results": pd.read_csv(CV_RESULTS_PATH),
        }
    data, X, y, train_idx, test_idx, feature_names = build_split_and_features()
    feature_sets = {
        "lasso_top15": select_top15_from_training_lasso(feature_names, X.iloc[train_idx], y.iloc[train_idx]),
    }
    model_specs = {
        "logistic_l2": (
            Pipeline([("scale", StandardScaler()), ("clf", ThresholdClassifier(LogisticRegression(penalty="l2", solver="liblinear", max_iter=5000, random_state=SEED)))]),
            {"clf__estimator__C": [0.001, 0.01, 0.1, 1.0, 10.0], "clf__threshold": THRESHOLDS},
        ),
        "knn": (
            Pipeline([("scale", StandardScaler()), ("clf", ThresholdClassifier(KNeighborsClassifier(metric="euclidean")))]),
            {"clf__estimator__n_neighbors": [3, 5, 9, 15, 25, 35], "clf__threshold": THRESHOLDS},
        ),
        "kernel_logistic_rbf": (
            Pipeline([("scale", StandardScaler()), ("kernel", Nystroem(kernel="rbf", random_state=SEED, n_components=200)), ("clf", ThresholdClassifier(LogisticRegression(penalty="l2", solver="liblinear", max_iter=5000, random_state=SEED)))]),
            {"kernel__gamma": [0.005, 0.01, 0.02, 0.05], "clf__estimator__C": [0.1, 1.0, 10.0], "clf__threshold": THRESHOLDS},
        ),
    }

    cv_rows, performance_rows = [], []
    predictions = data[["SerialNumber", "y_debt"]].copy()
    predictions["split"] = "test"
    predictions.loc[train_idx, "split"] = "train"

    for feature_set, cols in feature_sets.items():
        X_train, X_test = X.iloc[train_idx][cols], X.iloc[test_idx][cols]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        for model_name, (estimator, grid_params) in model_specs.items():
            grid, cv = fit_grid(model_name, estimator, grid_params, X_train, y_train)
            cv["feature_set"] = feature_set
            cv_rows.append(cv)
            train_pred = grid.predict(X_train)
            test_pred = grid.predict(X_test)
            train_score = grid.predict_proba(X_train)[:, 1]
            test_score = grid.predict_proba(X_test)[:, 1]
            prefix = f"{model_name}_{feature_set}"
            predictions[f"{prefix}_pred"] = np.nan
            predictions[f"{prefix}_score"] = np.nan
            predictions.loc[train_idx, f"{prefix}_pred"] = train_pred
            predictions.loc[test_idx, f"{prefix}_pred"] = test_pred
            predictions.loc[train_idx, f"{prefix}_score"] = train_score
            predictions.loc[test_idx, f"{prefix}_score"] = test_score

            performance_rows.append({
                "model": model_name,
                "feature_set": feature_set,
                "params": str(grid.best_params_),
                "mean_cv_f1": float(grid.best_score_),
                "train_f1": f1_score(y_train, train_pred),
                "test_f1": f1_score(y_test, test_pred),
                "n_train": len(train_idx),
                "n_test": len(test_idx),
                "n_raw_features": len(cols),
            })

    cv_results = pd.concat(cv_rows, ignore_index=True)
    performance = pd.DataFrame(performance_rows)
    safe_to_csv(cv_results, CV_RESULTS_PATH)
    safe_to_csv(performance, PERFORMANCE_PATH)
    safe_to_csv(predictions, PREDICTIONS_PATH)

    return {"performance": performance, "predictions": predictions, "cv_results": cv_results}

In [ ]:
part4_results = run_part4_models()
part4_results['performance']


In [ ]:
########## Auxiliary - supervised learning ##########

def plot_part4_results(results):
    performance = results["performance"]
    print(performance[["model", "feature_set", "params", "mean_cv_f1", "train_f1", "test_f1"]].to_string(index=False))
    x_pos = np.arange(len(performance))
    width = 0.35
    plt.figure(figsize=(11, 5))
    plt.bar(x_pos - width/2, performance["train_f1"], width, label="Train F1")
    plt.bar(x_pos + width/2, performance["test_f1"], width, label="Test F1")
    plt.xticks(x_pos, performance["model"], rotation=20, ha="right")
    plt.ylabel("F1")
    plt.title("Part 4 model F1 scores")
    plt.legend()
    plt.tight_layout()
    plt.savefig(F1_PLOT_PATH, dpi=140)
    plt.close()

plot_part4_results(part4_results)

### Q5 - Learner Diversity and Majority Vote


In [ ]:
########## Auxiliary - supervised learning ##########
ERRORS_PATH = os.path.join(SUPERVISED_DIR, 'learner_errors_part5.csv')
ERROR_MATRIX_PATH = os.path.join(SUPERVISED_DIR, 'learner_error_correlation_part5.csv')
MAJORITY_SUMMARY_PATH = os.path.join(SUPERVISED_DIR, 'majority_vote_summary_part5.csv')
MAJORITY_PREDICTIONS_PATH = os.path.join(SUPERVISED_DIR, 'majority_vote_predictions_part5.csv')


In [ ]:
## Step 5 - learner diversity and majority vote

def run_part5_learner_diversity():
    # Skip recomputing if outputs already exist
    if os.path.exists(ERRORS_PATH) and os.path.exists(MAJORITY_SUMMARY_PATH):
        return {
            "errors": pd.read_csv(ERRORS_PATH),
            "diversity_matrix": pd.read_csv(ERROR_MATRIX_PATH).set_index("learner"),
            "summary": pd.read_csv(MAJORITY_SUMMARY_PATH),
            "predictions": pd.read_csv(MAJORITY_PREDICTIONS_PATH),
        }
    predictions = pd.read_csv(PREDICTIONS_PATH)
    performance = pd.read_csv(PERFORMANCE_PATH)
    learners = ["logistic_l2", "knn", "kernel_logistic_rbf"]
    pred_cols = [f"{learner}_lasso_top15_pred" for learner in learners]
    train_mask = predictions["split"].eq("train")
    test_mask = predictions["split"].eq("test")

    errors = predictions.loc[train_mask, ["SerialNumber", "y_debt"]].copy()
    for learner, col in zip(learners, pred_cols):
        errors[f"{learner}_error"] = (predictions.loc[train_mask, col].astype(int).to_numpy() != predictions.loc[train_mask, "y_debt"].astype(int).to_numpy()).astype(int)

    E = errors[[f"{learner}_error" for learner in learners]].to_numpy(dtype=float)
    E_tilde = (E - E.mean(axis=0)) / np.sqrt(((E - E.mean(axis=0)) ** 2).sum(axis=0))  # we normalize column for cosine similarity
    diversity = pd.DataFrame(E_tilde.T @ E_tilde, index=learners, columns=learners)  # calculating gram matrix for error correlation
    diversity.index.name = "learner"

    majority_pred = (predictions[pred_cols].astype(int).sum(axis=1) >= 2).astype(int)  # we need 2/3 to agree
    summary_rows = []
    for learner, col in zip(learners, pred_cols):
        row = performance.loc[performance["model"].eq(learner)].iloc[0]
        summary_rows.append({"learner": learner, "train_f1": row["train_f1"], "test_f1": row["test_f1"], "combination": "base"})
    summary_rows.append({
        "learner": "majority_vote",
        "train_f1": f1_score_binary(predictions.loc[train_mask, "y_debt"], majority_pred[train_mask]),
        "test_f1": f1_score_binary(predictions.loc[test_mask, "y_debt"], majority_pred[test_mask]),
        "combination": "majority_vote",
    })
    summary = pd.DataFrame(summary_rows)
    best_base = summary.loc[summary["combination"].eq("base"), "test_f1"].max()
    summary["beats_best_base_test_f1"] = False
    summary.loc[summary["learner"].eq("majority_vote"), "beats_best_base_test_f1"] = summary.loc[summary["learner"].eq("majority_vote"), "test_f1"].iloc[0] > best_base

    out = predictions[["SerialNumber", "y_debt", "split"] + pred_cols].copy()
    out["majority_vote_pred"] = majority_pred
    out["majority_vote_error"] = (majority_pred != predictions["y_debt"].astype(int)).astype(int)

    safe_to_csv(errors, ERRORS_PATH)
    safe_to_csv(diversity.reset_index(), ERROR_MATRIX_PATH)
    safe_to_csv(summary, MAJORITY_SUMMARY_PATH)
    safe_to_csv(out, MAJORITY_PREDICTIONS_PATH)

    return {"errors": errors, "diversity_matrix": diversity, "summary": summary, "predictions": out}

In [ ]:
part5_results = run_part5_learner_diversity()
part5_results['summary']
print(part5_results["diversity_matrix"].round(4).to_string())
print(part5_results["summary"].to_string(index=False))

---
## Section 2 - Unsupervised Learning

### Q2 - Merging the crimes and cities tables

In [ ]:
########## Auxiliary - assisted by Claude ##########

def load_and_merge():
    CRIMES_FILE = os.path.join(DATA_DIR, '2012026154636_2025_hangashat_meyda_plili.xlsx')
    CITIES_FILE = os.path.join(DATA_DIR, 'cities_israel.xls')

    columns_to_read = ['YeshuvKod', 'StatisticGroupKod', 'StatisticTypeKod']
    df = pd.read_csv(CRIMES_FILE, usecols=columns_to_read, encoding='utf-8-sig')

    df_cities = pd.read_excel(CITIES_FILE, usecols=['symbol', 'coordinates', 'population'])
    df_cities = df_cities.rename(columns={'symbol': 'YeshuvKod'})

    df_counts = df.groupby(['YeshuvKod', 'StatisticTypeKod']).size().unstack(fill_value=0)
    df_merged = pd.merge(df_counts.reset_index(), df_cities, on='YeshuvKod', how='left')
    return df_merged

df_merged_raw = load_and_merge()
df_merged_raw.head(3)

### Q3 - Graph Laplacian

In [ ]:
########## Auxiliary - assisted by Claude ##########

def save_matrix(matrix, filename):
    with open(filename, 'wb') as f: pickle.dump(matrix, f)

def load_matrix(filename):
    with open(filename, 'rb') as f: return pickle.load(f)

We elaborated more on the preprocessing part in the writeup:

In [ ]:
########## Auxiliary - assisted by Claude ##########

BAD_GROUPS = {-1, 1200, 1300}

def preprocess(df_merged):
    # drop unknown settlement
    df_merged = df_merged.dropna(subset=['YeshuvKod'])

    # drop prblematic data categories as mentioned in the writeup: -1, 1200, 1300
    crime_cols_all = [c for c in df_merged.columns if isinstance(c, int) and c not in BAD_GROUPS]
    df_merged = df_merged[['YeshuvKod'] + crime_cols_all + ['coordinates', 'population']]

    # keep cities with known population and coords
    df_merged['population'] = pd.to_numeric(df_merged['population'], errors='coerce')
    df_merged['coordinates'] = pd.to_numeric(df_merged['coordinates'], errors='coerce')
    df_merged = df_merged.dropna(subset=['population', 'coordinates'])
    df_merged = df_merged[df_merged['population'] >= 2000]
    df_merged = df_merged.set_index('YeshuvKod')

    # Parse ITM coordinate integer into x / y
    coord_str = df_merged['coordinates'].astype(int).astype(str).str.zfill(10)
    df_merged['x'] = coord_str.str[:5].astype(float)
    df_merged['y'] = coord_str.str[5:10].astype(float)
    df_merged = df_merged.drop(columns=['coordinates'])

    crime_cols = [c for c in df_merged.columns if isinstance(c, int)]
    print(f'Cities after preprocessing: {len(df_merged)}')
    print(f'Crime subtypes: {len(crime_cols)}')
    return df_merged, crime_cols



Here we calculate the embedding matrix V, the weight matrix W and the diagonal matrix D, and finally the Laplacian L. The similarity and normalization logic we selected is in the writeup as well.

In [ ]:
## Step 3 - compute Laplacian

# Create embedding matrix:
def compute_embedding_matrix(C, pop):
    # noarmlize by population, resphape to (|crime_cols|, 1)
    V = C / pop[:, np.newaxis]
    # L2 normalization per row, so we compare crime mix profile
    row_norms = np.linalg.norm(V, axis=1, keepdims=True)
    row_norms[row_norms == 0] = 1
    V = V / row_norms
    return V

# calculating sigma based on knn like recommended in "A Tutorial on Spectral Clustering" paper we learned
def compute_sigmas(V, k=7):
    n = V.shape[0]
    sigmas = np.zeros(n)
    for i in range(n):
        dists = np.array([np.sum((V[i] - V[j]) ** 2) ** 0.5 for j in range(n) if j != i])
        dists.sort()
        sigmas[i] = dists[k - 1]
    return sigmas

# Compute weight matrix W:
def compute_LW_matrices(V, sigmas, n):
    W = np.zeros((n, n))
    for i in range(n):
        for j in range(i, n):
            diff = V[i]-V[j]
            # Kernel
            dist_squared =np.sum(diff**2)
            W[i,j] = np.exp(-dist_squared / (sigmas[i] * sigmas[j]))
            W[j,i] = W[i,j]
    np.fill_diagonal(W, 0)
    # Copmute L and D
    D= np.zeros((n,n))
    for i in range(n):
        cur_sum =0
        for j in range(n):
            cur_sum+= W[i,j]
        D[i,i] =  cur_sum
    L = D-W
    return L,W

df_proc, crime_cols = preprocess(df_merged_raw)
n   = len(df_proc)
C   = df_proc[crime_cols].values.astype(float)
pop = df_proc['population'].values
V   = compute_embedding_matrix(C, pop)
sigmas = compute_sigmas(V)

W_PATH = os.path.join(_CODE_DIR, 'W.pkl')
L_PATH = os.path.join(_CODE_DIR, 'L.pkl')
if os.path.exists(W_PATH) and os.path.exists(L_PATH):
    W = load_matrix(W_PATH)
    L = load_matrix(L_PATH)
else:
    L, W = compute_LW_matrices(V, sigmas, n)
    save_matrix(W, W_PATH)
    save_matrix(L, L_PATH)

### Q4 - Eigenvectors

In [ ]:
########## Auxiliary - assisted by Claude ##########

_itm_to_webmercator = Transformer.from_crs("EPSG:2039", "EPSG:3857", always_xy=True)

_LABELED_CITIES = [
    (19423, 38515, 'אילת'),
    (17966, 57357, 'באר שבע'),
    (22086, 63221, 'ירושלים'),
    (16691, 63377, 'אשדוד'),
    (18027, 66485, 'תל אביב-יפו'),
    (18726, 69145, 'נתניה'),
    (20112, 74546, 'חיפה'),
    (24942, 74369, 'טבריה'),
    (25383, 79074, 'קריית שמונה'),
]

def plot_cities(x, y, values, title, filename):
    mx, my = _itm_to_webmercator.transform(x * 10, y * 10)
    abs_max = np.percentile(np.abs(values), 95)
    norm = plt.Normalize(vmin=-abs_max, vmax=abs_max)
    fig, ax = plt.subplots(figsize=(8, 20))
    ax.scatter(mx, my, c=np.clip(values, -abs_max, abs_max), cmap='coolwarm', norm=norm,
               s=50, edgecolors='black', linewidths=0.3, alpha=0.9, zorder=3)
    for cx, cy, name in _LABELED_CITIES:
        wmx, wmy = _itm_to_webmercator.transform(cx * 10, cy * 10)
        ax.annotate(get_display(name), xy=(wmx, wmy), fontsize=12, rotation=90,
                    ha='center', va='bottom', zorder=4, color='#111111',
                    xytext=(0, 5), textcoords='offset points',
                    bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.6, ec='none'))
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldGrayCanvas, zoom=8)
    ax.set_axis_off()
    plt.tight_layout(pad=0)
    tmp_map = os.path.join(PLOTS_DIR_UNSUP, '_tmp_' + filename)
    plt.savefig(tmp_map, dpi=150, bbox_inches='tight')
    plt.close()
    rotated = Image.open(tmp_map).rotate(270, expand=True)
    os.remove(tmp_map)
    fig, (ax_map, ax_cb) = plt.subplots(1, 2, figsize=(20, 8),
                                         gridspec_kw={'width_ratios': [20, 1]})
    ax_map.imshow(rotated)
    ax_map.set_axis_off()
    ax_map.set_title(title, fontsize=16, pad=10)
    abs_max = np.percentile(np.abs(values), 95)
    norm = plt.Normalize(vmin=-abs_max, vmax=abs_max)
    sm = plt.cm.ScalarMappable(cmap='coolwarm', norm=norm)
    plt.colorbar(sm, cax=ax_cb)
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig(os.path.join(PLOTS_DIR_UNSUP, filename), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
## Step 4 - EV calculation
def calc_print_EV(L, x, y, city_names):
    # get eigenvectors and eigenvalues
    eigenvals, eigenvecs = np.linalg.eigh(L)

    for t in range(1, 11): # skipping the trivial 0-eigenvalue vector:
        ev = eigenvecs[:, t]
        title = f'Eigenvector {t} (λ={eigenvals[t]:.4f})'
        plot_cities(x, y, ev, title=title, filename=f'eigenvector_{t}.png')
    return eigenvals, eigenvecs

x, y = df_proc['x'].values, df_proc['y'].values
df_cities_names = pd.read_excel(os.path.join(DATA_DIR, 'cities_israel.xls'),
                                 usecols=['symbol', 'שם יישוב'])
kod_to_name = dict(zip(df_cities_names['symbol'], df_cities_names['שם יישוב']))
city_names = [kod_to_name.get(k, str(int(k))) for k in df_proc.index]

eigenvals, eigenvecs = calc_print_EV(L, x, y, city_names)

Here we can see the eigenvectors of each eigenvalue, sorted from largest to smallest. our analysis about which cities correspond to which vectors is in the writeup.

### Q5 - Spectral Clustering

In [ ]:
## Step 5 - spectral clustering
def spectral_clustering(eigenvecs, k):
    v = eigenvecs[:, 1:k + 1]
    kmeans = KMeans(n_clusters=k, random_state=23)
    labels = kmeans.fit_predict(v)
    return labels

clusters = spectral_clustering(eigenvecs, 5)

In [ ]:
########## Auxiliary - assisted by Claude ##########

def plot_eigengap(eigenvalues, n=20):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(range(1, n + 1), eigenvalues[1:n + 1], 'o-')
    ax.set_xlabel('Index')
    ax.set_ylabel('Eigenvalue')
    ax.set_title('Eigenvalue spectrum (eigengap)')
    ax.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR_UNSUP, 'eigengap.png'), dpi=150)
    plt.show()
    plt.close()

plot_eigengap(eigenvals)

def plot_clusters(x, y, labels, title, filename):
    mx, my = _itm_to_webmercator.transform(x * 10, y * 10)
    k = len(np.unique(labels))
    cmap = plt.colormaps.get_cmap('tab10').resampled(k)
    fig, ax = plt.subplots(figsize=(8, 20))
    ax.scatter(mx, my, c=labels, cmap=cmap, vmin=0, vmax=k - 1,
               s=50, edgecolors='black', linewidths=0.3, alpha=0.9, zorder=3)
    for cx, cy, name in _LABELED_CITIES:
        wmx, wmy = _itm_to_webmercator.transform(cx * 10, cy * 10)
        ax.annotate(get_display(name), xy=(wmx, wmy), fontsize=12, rotation=90,
                    ha='center', va='bottom', zorder=4, color='#111111',
                    xytext=(0, 5), textcoords='offset points',
                    bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.6, ec='none'))
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldGrayCanvas, zoom=8)
    ax.set_axis_off()
    plt.tight_layout(pad=0)
    tmp_map = os.path.join(PLOTS_DIR_UNSUP, '_tmp_' + filename)
    plt.savefig(tmp_map, dpi=150, bbox_inches='tight')
    plt.close()
    rotated = Image.open(tmp_map).rotate(270, expand=True)
    os.remove(tmp_map)
    fig, (ax_map, ax_cb) = plt.subplots(1, 2, figsize=(20, 8),
                                         gridspec_kw={'width_ratios': [20, 1]})
    ax_map.imshow(rotated)
    ax_map.set_axis_off()
    ax_map.set_title(title, fontsize=18)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=k - 1))
    cbar = plt.colorbar(sm, cax=ax_cb, ticks=range(k))
    cbar.set_ticklabels([f'Cluster {i+1}' for i in range(k)])
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR_UNSUP, filename), dpi=150)
    plt.show()
    plt.close()

plot_clusters(x, y, clusters, title='Spectral Clustering (k=5)', filename='clusters.png')

Here we plotted the eigengap graph and the spectral clusters for K=5. our selection of K=5 is justified in the writeup, and our analysis of the clusters as well.

### Q6 - Feature Correlations

In [ ]:
########## Auxiliary - assisted by Claude ##########

_df_religion = pd.read_excel(os.path.join(DATA_DIR, 'cities_israel.xls'),
                              usecols=['symbol', 'דת יישוב']).rename(columns={'symbol': 'YeshuvKod'})
_RELIGION_MAP = {1: 'Jewish', 2: 'Arab', 3: 'Druze/Other', 4: 'Mixed'}

_CITY_FEATURES = ['population', 'מחוז', 'דת יישוב', 'מעמד מונציפאלי', 'גובה ']
_FEATURE_LABELS = ['Population', 'District', 'Religion', 'Municipal status', 'Elevation']
_df_features = pd.read_excel(os.path.join(DATA_DIR, 'cities_israel.xls'),
                              usecols=['symbol'] + _CITY_FEATURES).rename(columns={'symbol': 'YeshuvKod'})

def print_correlation_table(labels, rows):
    print(f"{'Feature':<20}", end='')
    for label in labels:
        print(f"  {label:>6}", end='')
    print()
    print('-' * (20 + 8 * len(labels)))
    for feat_label, values, stars in rows:
        print(f"{feat_label:<20}", end='')
        for r, s in zip(values, stars):
            print(f"  {r:+.2f}{s}", end='')
        print()
    print()

In [ ]:
########## Auxiliary - assisted by Claude ##########

def analyze_ethnicity(clusters, yeshuv_kods):
    # pair city with its kod, join the religion df
    df = pd.DataFrame({'cluster': clusters, 'YeshuvKod': yeshuv_kods})
    df = df.merge(_df_religion, on='YeshuvKod', how='left')
    # convert codes to labels
    df['religion'] = df['דת יישוב'].map(_RELIGION_MAP).fillna('Unknown')
    # count cluster-religion pairs
    table = df.groupby(['cluster', 'religion']).size().unstack(fill_value=0)
    table.index = [f'Cluster {i+1}' for i in table.index]
    print(table.to_string())
    print()

def correlate_features(eigenvecs, yeshuv_kods):
    # load df
    df = pd.DataFrame({'YeshuvKod': yeshuv_kods}).merge(_df_features, on='YeshuvKod', how='left')
    ev_labels = [f'EV{t}' for t in range(1, 6)]
    rows = []
    for col, label in zip(_CITY_FEATURES, _FEATURE_LABELS):
        feat = df[col].values
        # mask missing values from correlation computation
        mask = ~np.isnan(feat)
        rs, stars = [], []
        # calculate spearman
        for t in range(1, 6):
            r, p = spearmanr(eigenvecs[mask, t], feat[mask])
            rs.append(r)
            # check for significance ( uncorrected for BH like in next part)
            stars.append('*' if p < 0.05 else ' ')
        rows.append((label, rs, stars))
    print_correlation_table(ev_labels, rows)

In [ ]:
## Step 6 - analyzing ethnicity and correlate eigenvectors with city features
analyze_ethnicity(clusters, list(df_proc.index))
correlate_features(eigenvecs, list(df_proc.index))

Results analysis is in the writeup

---
## Section 3 - Hypothesis Testing / Multiple Testing

Our Hypothesis testing was around differences in crime patterns between 2022 and 2025. Claude here assisted us with loading the tables and plotting the p-values. we used the crime rate per-year per-city as a statistic, normalized per-capita, and then performed standard two-sample t-testing. Afterwards we corrected for BH. Full explanation and analysis are in the writeup.

In [ ]:
########## Auxiliary - assisted by Claude ##########

def _load_one_file(filepath):
    cols = ['Year', 'YeshuvKod', 'StatisticGroupKod', 'StatisticTypeKod', 'StatisticType']
    # try csv first; fall back to xls/xlsx
    try:
        df = pd.read_csv(filepath, usecols=cols, encoding='utf-8-sig')
    except Exception:
        df = pd.read_excel(filepath, usecols=cols)
    return df

def load_yearly_rates(filepaths):
    # load files, filter bad groups and unknown cities, join population >= 2000
    frames = [_load_one_file(p) for p in filepaths]
    df = pd.concat(frames, ignore_index=True)
    df = df[df['Year'].isin([2022, 2025])]
    df = df[~df['StatisticGroupKod'].isin(BAD_GROUPS)]
    df = df.dropna(subset=['YeshuvKod'])

    df_cities = pd.read_excel(os.path.join(DATA_DIR, 'cities_israel.xls'),
                               usecols=['symbol', 'population'])
    df_cities = df_cities.rename(columns={'symbol': 'YeshuvKod'})
    df_cities['population'] = pd.to_numeric(df_cities['population'], errors='coerce')
    df_cities = df_cities[df_cities['population'] >= 2000]

    df = df.merge(df_cities[['YeshuvKod', 'population']], on='YeshuvKod', how='inner')
    return df

def plot_pvalues(df, bonf_threshold):
    # df is sorted output of run_t_test_with_bh (ascending by p_value)
    # style follows Ex2/q2.py: rank on x, p-value on log-scale y, horizontal threshold lines
    m = len(df)
    sorted_p = df['p_value'].values  # already sorted ascending
    bh_thresh = df.loc[df['reject_bh'], 'p_value'].max() if df['reject_bh'].any() else 0
    bonf_thresh = bonf_threshold

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(range(1, m + 1), sorted_p, marker='.', markersize=4, linewidth=0.8,
            color='steelblue', label='p-values')
    ax.axhline(bonf_thresh, color='orange', linestyle='--',
               label=f'Bonferroni ({bonf_thresh:.5f})')
    ax.axhline(bh_thresh, color='black', linestyle='--',
               label=f'BH threshold ({bh_thresh:.5f})')
    # list significant crime types in bottom-right as a text block
    sig = df[df['reject_bh']].reset_index(drop=True)
    label_text = 'Significant crime types:\n' + '\n'.join(
        get_display(f"{i+1}. {row['StatisticType']}")
        for i, row in sig.iterrows()
    )
    ax.text(0.98, 0.02, label_text, transform=ax.transAxes,
            fontsize=7, va='bottom', ha='right',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8, ec='lightgrey'))
    ax.set_yscale('log')
    ax.set_xlabel('Rank')
    ax.set_ylabel('p-value')
    ax.set_title('2022 vs 2025: crime type p-values')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR_HYP, 'pvalues.png'), dpi=150)
    plt.close()

In [ ]:
########## Multiple Hypothesis Testing ##########

def compute_city_rates(df):
    # aggregate (type, city, year)
    counts = (
        df.groupby(['StatisticTypeKod', 'StatisticType', 'YeshuvKod', 'population', 'Year'])
        .size()
        .rename('crime_count')
        .reset_index()
    )
    # compute per-capita crime rate
    counts['crime_rate'] = counts['crime_count'] / counts['population']
    return counts

def run_t_test(counts):
    results = []
    for type_kod, group in counts.groupby('StatisticTypeKod'):
        # each entry is one city's per-capita rate for the specific crime type
        rates_2022 = group[group['Year'] == 2022]['crime_rate'].values
        rates_2025 = group[group['Year'] == 2025]['crime_rate'].values

        # skip rare types with too few cities to estimate variance
        if len(rates_2022) < 2 or len(rates_2025) < 2:
            continue

        #calculating welch t_test statistic and accordingly the pval, without assuming equal varaince
        T_i, p_i = stats.ttest_ind(rates_2022, rates_2025, equal_var=False)
        results.append({
            'StatisticTypeKod': type_kod, 'StatisticType': group['StatisticType'].iloc[0], 'mean_2022': rates_2022.mean(),
            'mean_2025': rates_2025.mean(), 'effect': rates_2025.mean() - rates_2022.mean(), 'T_i': T_i, 'p_value': p_i,
        })
    return results

In [ ]:
## Step 3 - BH correction

def bh_correct(results, alpha=0.05):
    df = pd.DataFrame(results)
    # calc bonferroni
    m = len(df)
    bonf_threshold = alpha / m

    ##  calc BH:
    # sort pvals:
    df = df.sort_values('p_value').reset_index(drop=True)
    # calc bf threshold
    df['bh_threshold'] = (df.index + 1) * alpha / m
    # find k corresponding to threshold
    k = df[df['p_value'] <= df['bh_threshold']].index.max()
    # reject all hypotheses with rank <= k_max
    df['reject_bh'] = False
    if not pd.isna(k):
        df.loc[:k, 'reject_bh'] = True

    df['reject_bonferroni'] = df['p_value'] < bonf_threshold
    return df, bonf_threshold

In [ ]:
files = sorted(glob.glob(os.path.join(DATA_DIR, '2012026154636_2022*.csv'))) +         sorted(glob.glob(os.path.join(DATA_DIR, '2012026154636_2025*.xlsx')))
counts = compute_city_rates(load_yearly_rates(files))
results = run_t_test(counts)
df_results, bonf_threshold = bh_correct(results)
plot_pvalues(df_results, bonf_threshold)

sig_bh   = df_results[df_results['reject_bh']]
sig_bonf = df_results[df_results['reject_bonferroni']]
print(f"Total crime types tested: {len(df_results)}")
print(f"Bonferroni significant: {len(sig_bonf)}")
print(f"BH significant: {len(sig_bh)}")
sig_bh[['StatisticType', 'mean_2022', 'mean_2025', 'effect', 'p_value']]

---
## Section 4 - Transformers


### Part A - Architecture Design

We use the graph nodes as tokens. Each atom categorical feature is embedded separately and the embeddings are summed. Laplacian eigenvectors are added as positional encodings, and the transformer attention receives a learned bond/adjacency bias from the edge features.

```text
Molecular graph
-> atom nodes as tokens
-> categorical atom embeddings + Laplacian positional encoding
-> graph transformer blocks with bond/adjacency attention bias
-> masked mean+max graph pooling
-> MLP classifier
-> HIV inhibition probability
```


### Part B - ogbg-molhiv Experiment

This block can be run in two modes. The default is local quick mode, with `QUICK_RUN = True`, so the notebook only uses a small subset and 2 epochs for a fast sanity check. For the full Colab run, switch to a GPU runtime, set `QUICK_RUN = False`, and keep `SAVE_OUTPUTS_TO_GOOGLE_DRIVE = True`.

In Colab, the notebook creates and saves outputs under:

```text
/content/drive/MyDrive/ModernStatisticsFinalProject/transformers
```

This folder contains the result tables, plots, Laplacian-PE cache, OGB data, and model weights.


In [ ]:
########## Auxiliary - transformer setup ##########

INSTALL_TRANSFORMER_PACKAGES = False

if INSTALL_TRANSFORMER_PACKAGES:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ogb', 'torch-geometric', 'tqdm'])

import copy, math, random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from ogb.graphproppred import Evaluator, PygGraphPropPredDataset
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# PyTorch 2.6 loads with weights_only=True by default; OGB/PyG cached data needs the older behavior.
try:
    from torch.serialization import add_safe_globals
    from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
    from torch_geometric.data.storage import GlobalStorage
    add_safe_globals([DataEdgeAttr, DataTensorAttr, GlobalStorage])
except Exception as exc:
    print('PyG safe-global registration skipped:', exc)

if not hasattr(torch, '_original_load_before_weights_only_patch'):
    torch._original_load_before_weights_only_patch = torch.load


def torch_load_compat(*args, **kwargs):
    if 'weights_only' not in kwargs:
        kwargs['weights_only'] = False
    return torch._original_load_before_weights_only_patch(*args, **kwargs)


torch.load = torch_load_compat

SAVE_OUTPUTS_TO_GOOGLE_DRIVE = True
GOOGLE_DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/ModernStatisticsFinalProject/transformers')


def running_in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


if SAVE_OUTPUTS_TO_GOOGLE_DRIVE and running_in_colab():
    from google.colab import drive
    drive.mount('/content/drive')
    TRANSFORMER_OUTPUT_ROOT = GOOGLE_DRIVE_OUTPUT_DIR
else:
    TRANSFORMER_OUTPUT_ROOT = Path(DATA_DIR).parent

TRANSFORMER_DATA_DIR = TRANSFORMER_OUTPUT_ROOT / 'data' / 'transformers' if not running_in_colab() else TRANSFORMER_OUTPUT_ROOT / 'data'
TRANSFORMER_PLOTS_DIR = TRANSFORMER_OUTPUT_ROOT / 'plots' / 'transformers' if not running_in_colab() else TRANSFORMER_OUTPUT_ROOT / 'plots'
TRANSFORMER_CACHE_DIR = TRANSFORMER_OUTPUT_ROOT / 'cache'
TRANSFORMER_CHECKPOINT_DIR = TRANSFORMER_OUTPUT_ROOT / 'checkpoints'
TRANSFORMER_OGB_ROOT = TRANSFORMER_OUTPUT_ROOT / 'ogb'

for path in [TRANSFORMER_DATA_DIR, TRANSFORMER_PLOTS_DIR, TRANSFORMER_CACHE_DIR, TRANSFORMER_CHECKPOINT_DIR, TRANSFORMER_OGB_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print('Transformer output root:', TRANSFORMER_OUTPUT_ROOT)
print('Stats folder:', TRANSFORMER_DATA_DIR)
print('Plots folder:', TRANSFORMER_PLOTS_DIR)
print('Weights folder:', TRANSFORMER_CHECKPOINT_DIR)


In [ ]:
## Step B1 - config and data loading

TRANSFORMER_SEED = 42
QUICK_RUN = True  # set False for full run on Colab GPU

BATCH_SIZE = 64
EPOCHS = 20
LR = 1e-3
WEIGHT_DECAY = 1e-5

D_MODEL = 64
NUM_LAYERS = 3
NUM_HEADS = 4
FFN_DIM = 128
DROPOUT = 0.10
LAP_PE_DIM = 8
MAX_NODES = None

QUICK_EPOCHS = 2
QUICK_TRAIN_GRAPHS = 512
QUICK_VALID_GRAPHS = 256
QUICK_TEST_GRAPHS = 256
QUICK_MAX_TRAIN_BATCHES = 8
QUICK_MAX_EVAL_BATCHES = 4

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def set_transformer_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_transformer_seed(TRANSFORMER_SEED)

molhiv_dataset = PygGraphPropPredDataset(name='ogbg-molhiv', root=str(TRANSFORMER_OGB_ROOT))
molhiv_evaluator = Evaluator(name='ogbg-molhiv')
split_idx = molhiv_dataset.get_idx_split()

train_idx = split_idx['train']
valid_idx = split_idx['valid']
test_idx = split_idx['test']

if QUICK_RUN:
    train_idx = train_idx[:QUICK_TRAIN_GRAPHS]
    valid_idx = valid_idx[:QUICK_VALID_GRAPHS]
    test_idx = test_idx[:QUICK_TEST_GRAPHS]

example_graph = molhiv_dataset[0]


def compute_feature_cardinalities(dataset):
    node_max, edge_max = None, None
    for data in tqdm(dataset, desc='Scanning feature cardinalities'):
        cur_node_max = data.x.long().max(dim=0).values
        cur_edge_max = data.edge_attr.long().max(dim=0).values if data.edge_attr.numel() else torch.zeros(3, dtype=torch.long)
        node_max = cur_node_max if node_max is None else torch.maximum(node_max, cur_node_max)
        edge_max = cur_edge_max if edge_max is None else torch.maximum(edge_max, cur_edge_max)
    return (node_max + 1).tolist(), (edge_max + 1).tolist()


node_cardinalities, edge_cardinalities = compute_feature_cardinalities(molhiv_dataset)

In [ ]:
########## Auxiliary - Laplacian PE and padded graph batches ##########

def compute_laplacian_pe(edge_index, num_nodes, pe_dim):
    if num_nodes <= 0:
        return torch.zeros((0, pe_dim), dtype=torch.float32)

    adjacency = torch.zeros((num_nodes, num_nodes), dtype=torch.float32, device=edge_index.device)
    if edge_index.numel() > 0:
        src, dst = edge_index[0], edge_index[1]
        adjacency[src, dst] = 1.0
        adjacency[dst, src] = 1.0

    degree = adjacency.sum(dim=1)
    inv_sqrt_degree = torch.zeros_like(degree)
    positive_degree = degree > 0
    inv_sqrt_degree[positive_degree] = degree[positive_degree].pow(-0.5)

    normalized_adjacency = inv_sqrt_degree[:, None] * adjacency * inv_sqrt_degree[None, :]
    laplacian = torch.eye(num_nodes, dtype=torch.float32, device=edge_index.device) - normalized_adjacency
    _, eigenvectors = torch.linalg.eigh(laplacian)
    pe = eigenvectors[:, 1:1 + pe_dim] if num_nodes > 1 else eigenvectors[:, :0]

    if pe.shape[1] < pe_dim:
        pe = torch.cat([pe, torch.zeros((num_nodes, pe_dim - pe.shape[1]), device=edge_index.device)], dim=1)

    for j in range(pe.shape[1]):
        col = pe[:, j]
        if col.numel() and col[col.abs().argmax()] < 0:
            pe[:, j] = -col
    return pe.cpu().float()


class LaplacianPECache:
    def __init__(self, dataset, pe_dim, cache_path):
        self.dataset = dataset
        self.pe_dim = pe_dim
        self.cache_path = Path(cache_path)
        self.cache = torch.load(self.cache_path, map_location='cpu') if self.cache_path.exists() else {}

    def get(self, idx):
        idx = int(idx)
        if idx not in self.cache:
            data = self.dataset[idx]
            self.cache[idx] = compute_laplacian_pe(data.edge_index, data.num_nodes, self.pe_dim)
        return self.cache[idx]

    def save(self):
        self.cache_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(self.cache, self.cache_path)

    def precompute(self, indices):
        for idx in tqdm(indices, desc='Precomputing Laplacian PE'):
            self.get(int(idx))
        self.save()


pe_cache = LaplacianPECache(molhiv_dataset, LAP_PE_DIM, TRANSFORMER_CACHE_DIR / f'lap_pe_dim_{LAP_PE_DIM}.pt')
pe_cache.precompute(torch.cat([train_idx, valid_idx, test_idx]).tolist())


class IndexedGraphDataset(Dataset):
    def __init__(self, dataset, indices):
        self.dataset = dataset
        self.indices = [int(i) for i in indices]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        idx = self.indices[position]
        return self.dataset[idx], idx


def collate_graphs(batch):
    graphs, indices = zip(*batch)
    batch_size = len(graphs)
    node_counts = [int(g.num_nodes) for g in graphs]
    max_nodes = max(node_counts) if MAX_NODES is None else min(max(node_counts), MAX_NODES)

    x_cat = torch.zeros((batch_size, max_nodes, graphs[0].x.shape[1]), dtype=torch.long)
    y = torch.zeros((batch_size, 1), dtype=torch.float32)
    node_mask = torch.zeros((batch_size, max_nodes), dtype=torch.bool)
    lap_pe = torch.zeros((batch_size, max_nodes, LAP_PE_DIM), dtype=torch.float32)
    edge_attr_dense = torch.zeros((batch_size, max_nodes, max_nodes, graphs[0].edge_attr.shape[1]), dtype=torch.long)
    edge_mask = torch.zeros((batch_size, max_nodes, max_nodes), dtype=torch.bool)
    adjacency_mask = torch.zeros((batch_size, max_nodes, max_nodes), dtype=torch.bool)

    for b, (graph, idx) in enumerate(zip(graphs, indices)):
        n = min(int(graph.num_nodes), max_nodes)
        x_cat[b, :n] = graph.x[:n].long()
        y[b] = graph.y.view(1).float()
        node_mask[b, :n] = True
        lap_pe[b, :n] = pe_cache.get(idx)[:n]

        edge_index = graph.edge_index.long()
        if edge_index.numel() > 0:
            src, dst = edge_index[0], edge_index[1]
            keep = (src < n) & (dst < n)
            src, dst = src[keep], dst[keep]
            edge_attr_dense[b, src, dst] = graph.edge_attr.long()[keep]
            edge_mask[b, src, dst] = True
            adjacency_mask[b, src, dst] = True
            adjacency_mask[b, dst, src] = True

    return {
        'x_cat': x_cat,
        'y': y,
        'node_mask': node_mask,
        'lap_pe': lap_pe,
        'edge_attr_dense': edge_attr_dense,
        'edge_mask': edge_mask,
        'adjacency_mask': adjacency_mask,
        'graph_idx': torch.tensor(indices, dtype=torch.long),
    }


train_loader = DataLoader(IndexedGraphDataset(molhiv_dataset, train_idx), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_graphs)
valid_loader = DataLoader(IndexedGraphDataset(molhiv_dataset, valid_idx), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_graphs)
test_loader = DataLoader(IndexedGraphDataset(molhiv_dataset, test_idx), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_graphs)


In [ ]:
########## Auxiliary - graph transformer model ##########

class CategoricalNodeEncoder(nn.Module):
    def __init__(self, cardinalities, d_model):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(int(card), d_model) for card in cardinalities])

    def forward(self, x_cat):
        out = 0
        for j, emb in enumerate(self.embeddings):
            out = out + emb(x_cat[:, :, j])
        return out


class EdgeAttentionBias(nn.Module):
    def __init__(self, edge_cardinalities, num_heads):
        super().__init__()
        self.edge_embeddings = nn.ModuleList([nn.Embedding(int(card), num_heads) for card in edge_cardinalities])
        self.adjacency_bias = nn.Parameter(torch.zeros(num_heads))

    def forward(self, edge_attr_dense, edge_mask, adjacency_mask):
        bias = 0
        for j, emb in enumerate(self.edge_embeddings):
            bias = bias + emb(edge_attr_dense[:, :, :, j])
        bias = bias.permute(0, 3, 1, 2) * edge_mask.unsqueeze(1).float()
        bias = bias + self.adjacency_bias.view(1, -1, 1, 1) * adjacency_mask.unsqueeze(1).float()
        return bias


class GraphMultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, node_mask, attention_bias):
        bsz, num_nodes, _ = x.shape
        q = self.q_proj(x).view(bsz, num_nodes, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(bsz, num_nodes, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(bsz, num_nodes, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores + attention_bias
        scores = scores.masked_fill(~node_mask[:, None, None, :], torch.finfo(scores.dtype).min)
        attn = self.dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(bsz, num_nodes, self.d_model)
        out = self.out_proj(out)
        return out * node_mask.unsqueeze(-1).float()


class GraphTransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, ffn_dim, dropout):
        super().__init__()
        self.attention = GraphMultiHeadSelfAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(ffn_dim, d_model))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, node_mask, attention_bias):
        x = self.norm1(x + self.dropout(self.attention(x, node_mask, attention_bias)))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x * node_mask.unsqueeze(-1).float()


class GraphTransformerClassifier(nn.Module):
    def __init__(self, node_cardinalities, edge_cardinalities):
        super().__init__()
        self.node_encoder = CategoricalNodeEncoder(node_cardinalities, D_MODEL)
        self.pe_encoder = nn.Linear(LAP_PE_DIM, D_MODEL)
        self.edge_bias = EdgeAttentionBias(edge_cardinalities, NUM_HEADS)
        self.blocks = nn.ModuleList([GraphTransformerBlock(D_MODEL, NUM_HEADS, FFN_DIM, DROPOUT) for _ in range(NUM_LAYERS)])
        self.head = nn.Sequential(nn.LayerNorm(2 * D_MODEL), nn.Linear(2 * D_MODEL, D_MODEL), nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(D_MODEL, 1))

    def forward(self, batch):
        x = self.node_encoder(batch['x_cat']) + self.pe_encoder(batch['lap_pe'])
        x = x * batch['node_mask'].unsqueeze(-1).float()
        attention_bias = self.edge_bias(batch['edge_attr_dense'], batch['edge_mask'], batch['adjacency_mask'])
        for block in self.blocks:
            x = block(x, batch['node_mask'], attention_bias)

        mask = batch['node_mask'].unsqueeze(-1)
        mean_pool = (x * mask.float()).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        max_pool = x.masked_fill(~mask, torch.finfo(x.dtype).min).max(dim=1).values
        return self.head(torch.cat([mean_pool, max_pool], dim=-1))


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [ ]:
########## Auxiliary - training and evaluation ##########

def move_batch_to_device(batch, device):
    return {key: value.to(device) if torch.is_tensor(value) else value for key, value in batch.items()}


def labels_mask(y):
    return ~torch.isnan(y.view(-1))


def roc_auc_from_arrays(y_true, y_pred):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(molhiv_evaluator.eval({'y_true': y_true.reshape(-1, 1), 'y_pred': y_pred.reshape(-1, 1)})['rocauc'])


def batch_accuracy(logits, y):
    mask = labels_mask(y)
    probs = torch.sigmoid(logits.view(-1)[mask])
    preds = (probs >= 0.5).long()
    target = y.view(-1)[mask].long()
    return float((preds == target).float().mean().item()), int(mask.sum())


def train_one_epoch(model, loader, optimizer, criterion, max_batches=None):
    model.train()
    total_loss, total_examples, acc_sum = 0.0, 0, 0.0
    y_true_all, y_pred_all = [], []
    for batch_idx, batch in enumerate(tqdm(loader, desc='train', leave=False)):
        if max_batches is not None and batch_idx >= max_batches:
            break
        batch = move_batch_to_device(batch, DEVICE)
        y = batch['y']
        mask = labels_mask(y)
        optimizer.zero_grad()
        logits = model(batch)
        loss = criterion(logits.view(-1)[mask], y.view(-1)[mask])
        loss.backward()
        optimizer.step()

        n = int(mask.sum())
        total_loss += float(loss.item()) * n
        total_examples += n
        acc, acc_n = batch_accuracy(logits.detach(), y)
        acc_sum += acc * acc_n
        y_true_all.append(y.view(-1)[mask].detach().cpu().numpy())
        y_pred_all.append(torch.sigmoid(logits.view(-1)[mask]).detach().cpu().numpy())

    y_true = np.concatenate(y_true_all)
    y_pred = np.concatenate(y_pred_all)
    return {'loss': total_loss / total_examples, 'accuracy': acc_sum / total_examples, 'rocauc': roc_auc_from_arrays(y_true, y_pred)}


@torch.no_grad()
def evaluate(model, loader, criterion, max_batches=None, desc='eval'):
    model.eval()
    total_loss, total_examples, acc_sum = 0.0, 0, 0.0
    y_true_all, y_pred_all = [], []
    for batch_idx, batch in enumerate(tqdm(loader, desc=desc, leave=False)):
        if max_batches is not None and batch_idx >= max_batches:
            break
        batch = move_batch_to_device(batch, DEVICE)
        y = batch['y']
        mask = labels_mask(y)
        logits = model(batch)
        loss = criterion(logits.view(-1)[mask], y.view(-1)[mask])

        n = int(mask.sum())
        total_loss += float(loss.item()) * n
        total_examples += n
        acc, acc_n = batch_accuracy(logits, y)
        acc_sum += acc * acc_n
        y_true_all.append(y.view(-1)[mask].detach().cpu().numpy())
        y_pred_all.append(torch.sigmoid(logits.view(-1)[mask]).detach().cpu().numpy())

    y_true = np.concatenate(y_true_all)
    y_pred = np.concatenate(y_pred_all)
    return {'loss': total_loss / total_examples, 'accuracy': acc_sum / total_examples, 'rocauc': roc_auc_from_arrays(y_true, y_pred)}


In [ ]:
########## Auxiliary - transformer setup ##########

def run_mean_max_transformer():
    set_transformer_seed(TRANSFORMER_SEED)
    model_name = 'graph_transformer_mean_max_pool'
    model = GraphTransformerClassifier(node_cardinalities, edge_cardinalities).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss()

    epochs = QUICK_EPOCHS if QUICK_RUN else EPOCHS
    max_train_batches = QUICK_MAX_TRAIN_BATCHES if QUICK_RUN else None
    max_eval_batches = QUICK_MAX_EVAL_BATCHES if QUICK_RUN else None
    best_valid_rocauc = -np.inf
    best_epoch, best_state = None, None
    history_rows = []

    print(f'{model_name}: {count_parameters(model):,} trainable parameters')
    for epoch in range(1, epochs + 1):
        train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, max_batches=max_train_batches)
        valid_metrics = evaluate(model, valid_loader, criterion, max_batches=max_eval_batches, desc='valid')
        row = {
            'model_name': model_name,
            'epoch': epoch,
            'train_loss': train_metrics['loss'],
            'valid_loss': valid_metrics['loss'],
            'train_accuracy': train_metrics['accuracy'],
            'valid_accuracy': valid_metrics['accuracy'],
            'train_rocauc': train_metrics['rocauc'],
            'valid_rocauc': valid_metrics['rocauc'],
        }
        history_rows.append(row)
        print(f"epoch {epoch:02d} | train loss {row['train_loss']:.4f} acc {row['train_accuracy']:.4f} auc {row['train_rocauc']:.4f} | valid loss {row['valid_loss']:.4f} acc {row['valid_accuracy']:.4f} auc {row['valid_rocauc']:.4f}")

        if row['valid_rocauc'] > best_valid_rocauc:
            best_valid_rocauc = row['valid_rocauc']
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    test_metrics = evaluate(model, test_loader, criterion, max_batches=max_eval_batches, desc='test')
    history = pd.DataFrame(history_rows)
    results = pd.DataFrame([{
        'model_name': model_name,
        'pool_type': 'mean_max',
        'best_epoch': best_epoch,
        'best_valid_rocauc': best_valid_rocauc,
        'test_rocauc_at_best_valid': test_metrics['rocauc'],
        'final_train_loss': history.iloc[-1]['train_loss'],
        'final_valid_loss': history.iloc[-1]['valid_loss'],
        'parameter_count': count_parameters(model),
    }])

    history_path = TRANSFORMER_DATA_DIR / 'graph_transformer_history.csv'
    results_path = TRANSFORMER_DATA_DIR / 'graph_transformer_results.csv'
    checkpoint_path = TRANSFORMER_CHECKPOINT_DIR / f'{model_name}.pt'
    history.to_csv(history_path, index=False)
    results.to_csv(results_path, index=False)
    torch.save({
        'model_state_dict': model.state_dict(),
        'best_epoch': best_epoch,
        'best_valid_rocauc': best_valid_rocauc,
        'node_cardinalities': node_cardinalities,
        'edge_cardinalities': edge_cardinalities,
        'config': {'d_model': D_MODEL, 'num_layers': NUM_LAYERS, 'num_heads': NUM_HEADS, 'ffn_dim': FFN_DIM, 'dropout': DROPOUT, 'lap_pe_dim': LAP_PE_DIM},
    }, checkpoint_path)

    pe_cache.save()
    print('Saved history to:', history_path)
    print('Saved results to:', results_path)
    print('Saved model weights to:', checkpoint_path)
    return model, history, results

In [ ]:
## Step B2 - train the selected mean+max model
transformer_model, transformer_history, transformer_results = run_mean_max_transformer()
transformer_results

In [ ]:
########## Auxiliary - transformer setup ##########

def plot_transformer_history(history):
    model_name = history['model_name'].iloc[0]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(history['epoch'], history['train_loss'], marker='o', label='train')
    ax.plot(history['epoch'], history['valid_loss'], marker='o', label='valid')
    ax.set_title('Graph transformer mean+max: loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('BCE loss')
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.savefig(TRANSFORMER_PLOTS_DIR / f'{model_name}_loss.png', dpi=150)
    plt.show()
    plt.close()

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(history['epoch'], history['train_accuracy'], marker='o', label='train')
    ax.plot(history['epoch'], history['valid_accuracy'], marker='o', label='valid')
    ax.set_title('Graph transformer mean+max: accuracy')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.savefig(TRANSFORMER_PLOTS_DIR / f'{model_name}_accuracy.png', dpi=150)
    plt.show()
    plt.close()

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(history['epoch'], history['valid_rocauc'], marker='o', color='tab:green')
    ax.set_title('Graph transformer mean+max: validation ROC-AUC')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('ROC-AUC')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(TRANSFORMER_PLOTS_DIR / f'{model_name}_valid_rocauc.png', dpi=150)
    plt.show()
    plt.close()

In [ ]:
## Step B3 - plot train/validation curves
plot_transformer_history(transformer_history)